# 04a — Prompting infrastructure (shared)

**Project:** Lightweight domain adaptation of small LLMs for chemistry using LoRA and QLoRA (MSc FYP).

**What Phase 4 is doing:** evaluating off-the-shelf LLMs on BBBP / BACE / ESOL using *prompting only* — no fine-tuning. This is the real anchor for the FYP claim: "PEFT helps because plain prompting doesn't."

**This notebook (04a):** builds and validates the shared infrastructure that 04b and 04c will reuse. We don't run full evaluations here — we *smoke-test* the components on 10 BBBP rows with GPT-2 so we catch bugs before spending hours on Phi-4-mini.

**What we build:**
1. Dataset configuration (task type, label words / buckets, primary metric)
2. Prompt templates per dataset
3. Logit-scoring function (probability over label tokens, no generation)
4. ESOL bucketisation (regression → discrete buckets for logit-scoring)
5. Tanimoto retrieval index over training Morgan fingerprints
6. Exemplar samplers (random K-shot + retrieval K-shot)
7. Smoke test: GPT-2 + 10 BBBP rows under zero / few / retrieval

**This notebook does not write to `baselines.csv`** — that's 04b and 04c's job. This one just proves the pieces work.

## 1. Install + imports

In [ ]:
%pip install -q transformers rdkit pandas scikit-learn

In [ ]:
import os, random
from dataclasses import dataclass
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import AllChem
from sklearn.metrics import roc_auc_score, f1_score
from transformers import AutoTokenizer, AutoModelForCausalLM

RDLogger.DisableLog("rdApp.*")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU   :", torch.cuda.get_device_name(0))

## 2. Mount Drive + load Phase 02 splits

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/chemistry-peft-fyp/data"

def load_splits(name):
    prefix = name.lower()
    tr = pd.read_csv(f"{DATA_DIR}/{prefix}_train.csv")
    te = pd.read_csv(f"{DATA_DIR}/{prefix}_test.csv")
    return tr, te

splits = {name: load_splits(name) for name in ["BBBP", "BACE", "ESOL"]}
for n, (tr, te) in splits.items():
    print(f"{n:5s}: train={len(tr)}  test={len(te)}")

## 3. Dataset configuration

Per-dataset settings that drive prompt construction and evaluation. Classification = yes/no logit scoring. Regression (ESOL) = bucket-name logit scoring (MolReGPT-style), then convert back to a continuous prediction using the bucket midpoints.

In [ ]:
# ESOL solubility buckets (log mol/L).
# Boundaries chosen to roughly quintile-split the ESOL training distribution.
ESOL_BUCKETS = [
    ("very_low",  -np.inf, -6.0, -7.5),
    ("low",       -6.0,    -4.0, -5.0),
    ("medium",    -4.0,    -2.0, -3.0),
    ("high",      -2.0,     0.0, -1.0),
    ("very_high",  0.0,    np.inf, 1.0),
]
ESOL_LABEL_WORDS = [b[0] for b in ESOL_BUCKETS]
ESOL_MIDPOINTS   = np.array([b[3] for b in ESOL_BUCKETS], dtype=np.float32)

def esol_continuous_to_bucket(val):
    """Map a log-solubility value to one of the 5 bucket label words."""
    for label, lo, hi, _ in ESOL_BUCKETS:
        if lo < val <= hi:
            return label
    return ESOL_BUCKETS[0][0]  # fallback for val == -inf

@dataclass
class DatasetConfig:
    name: str
    task: str                       # "classification" or "regression"
    question: str                   # human-readable question for the prompt
    label_words: List[str]          # tokens to score (e.g. ["yes", "no"])
    positive_word: Optional[str]    # for classification, which word means y=1
    primary_metric: str             # "roc_auc" or "rmse"

DATASETS = {
    "BBBP": DatasetConfig(
        name="BBBP", task="classification",
        question="Does this molecule cross the blood-brain barrier?",
        label_words=["yes", "no"], positive_word="yes",
        primary_metric="roc_auc",
    ),
    "BACE": DatasetConfig(
        name="BACE", task="classification",
        question="Does this molecule inhibit the BACE-1 enzyme?",
        label_words=["yes", "no"], positive_word="yes",
        primary_metric="roc_auc",
    ),
    "ESOL": DatasetConfig(
        name="ESOL", task="regression",
        question="What is the aqueous solubility of this molecule?",
        label_words=ESOL_LABEL_WORDS, positive_word=None,
        primary_metric="rmse",
    ),
}
for cfg in DATASETS.values():
    print(cfg)

## 4. Prompt templates

Same skeleton for all three prompting regimes — only the *examples* block differs (empty for zero-shot, random for few-shot, retrieved for retrieval).

We end the prompt with `\nAnswer:` and score the next-token distribution. No generation.

In [ ]:
def label_for_prompt(cfg: DatasetConfig, raw_label):
    """Convert a row's raw label into the word that should appear in the prompt example."""
    if cfg.task == "classification":
        return cfg.positive_word if int(raw_label) == 1 else ("no" if cfg.positive_word == "yes" else "0")
    # regression → bucket name
    return esol_continuous_to_bucket(float(raw_label))

def build_prompt(cfg: DatasetConfig, test_smiles: str, exemplars: List[Tuple[str, str]] = None) -> str:
    """Construct a prompt with optional in-context exemplars.

    exemplars: list of (smiles, label_word) pairs to include before the test molecule.
    """
    parts = []
    if exemplars:
        for smi, lab in exemplars:
            parts.append(f"{cfg.question}\nSMILES: {smi}\nAnswer: {lab}\n")
    parts.append(f"{cfg.question}\nSMILES: {test_smiles}\nAnswer:")
    return "\n".join(parts)

# Show a sample prompt
sample = build_prompt(
    DATASETS["BBBP"],
    test_smiles="CC(=O)Oc1ccccc1C(=O)O",
    exemplars=[("CCO", "yes"), ("c1ccncc1", "no")],
)
print(sample)

## 5. Logit-scoring (no generation)

The crux of Phase 4. Instead of asking the LLM to generate text and parsing the answer, we ask the LLM for the probability distribution over the *very next token*, then look at the mass on our label words.

**Why this is robust:** the LLM can't refuse, can't ramble, can't get cut off. We get a number between 0 and 1 for every test molecule — perfect for ROC-AUC.

**Subtlety:** different tokenizers split words differently. " yes" might be one token in GPT-2 but two in SmolLM3. We use a small helper that finds the token-id for the *first* sub-token of each label word, which is what the model decides at the answer position.

Note that we score the label token without a leading space too — some tokenizers prepend space, some don't. We try both and take the more likely variant.

In [ ]:
def label_token_ids(tokenizer, label_words: List[str]) -> List[List[int]]:
    """For each label word, return the token ids of its 'with-space' and 'no-space' first tokens."""
    ids_per_label = []
    for w in label_words:
        candidates = set()
        for variant in (" " + w, w):
            t = tokenizer(variant, add_special_tokens=False).input_ids
            if t:
                candidates.add(t[0])
        ids_per_label.append(sorted(candidates))
    return ids_per_label

@torch.no_grad()
def score_labels(model, tokenizer, prompt: str, label_words: List[str]) -> np.ndarray:
    """Return a probability array over label_words, normalised to sum to 1."""
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    out = model(**enc)
    next_logits = out.logits[0, -1, :]                  # logits for the next token
    log_probs = torch.log_softmax(next_logits, dim=-1)

    token_ids_per_label = label_token_ids(tokenizer, label_words)
    label_logps = []
    for ids in token_ids_per_label:
        # Take max log-prob across the with-space / no-space variants for this label word.
        best = max(log_probs[i].item() for i in ids)
        label_logps.append(best)

    logps = np.array(label_logps, dtype=np.float64)
    # Re-normalise across just the label words (softmax over the subset).
    logps -= logps.max()
    probs = np.exp(logps)
    probs /= probs.sum()
    return probs

## 6. Tanimoto retrieval index

For the retrieval regime: given a test molecule, find the K most structurally similar molecules in the **training** set, and use those as in-context examples.

**Similarity metric:** Tanimoto on Morgan fingerprints (radius 2, 2048 bits). The cheminformatics default — same featurisation as Phase 03a but used for similarity search instead of as RF input.

**Why retrieval helps (MolReGPT thesis):** LLMs are better at "this molecule looks like *these known molecules*" than "answer in the abstract." Similar exemplars > random exemplars.

In [ ]:
_FPGEN = AllChem.GetMorganGenerator(radius=2, fpSize=2048)

def smiles_to_bitvect(smi: str):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    return _FPGEN.GetFingerprint(mol)

class TanimotoIndex:
    """Brute-force Tanimoto similarity search over a training set."""
    def __init__(self, train_df: pd.DataFrame):
        self.smiles = train_df["smiles"].tolist()
        self.labels = train_df["label"].tolist()
        self.fps    = []
        self.valid_idx = []
        for i, s in enumerate(self.smiles):
            fp = smiles_to_bitvect(s)
            if fp is not None:
                self.fps.append(fp)
                self.valid_idx.append(i)
        print(f"Indexed {len(self.fps)} / {len(self.smiles)} training molecules")

    def topk(self, query_smiles: str, k: int = 5) -> List[Tuple[str, float, object]]:
        """Return up to k (smiles, label, similarity) tuples, most similar first."""
        q = smiles_to_bitvect(query_smiles)
        if q is None:
            return []
        sims = DataStructs.BulkTanimotoSimilarity(q, self.fps)
        order = np.argsort(sims)[::-1][:k]
        return [(self.smiles[self.valid_idx[i]], self.labels[self.valid_idx[i]], float(sims[i])) for i in order]

# Build an index on BBBP train for the smoke test
bbbp_train, bbbp_test = splits["BBBP"]
bbbp_index = TanimotoIndex(bbbp_train)
demo = bbbp_index.topk(bbbp_test["smiles"].iloc[0], k=3)
print("\nNearest 3 training molecules to test[0]:")
for smi, lab, sim in demo:
    print(f"  sim={sim:.3f}  label={lab}  {smi[:80]}")

## 7. Exemplar samplers (random + retrieval)

Two functions that return a list of `(smiles, label_word)` exemplars suitable for `build_prompt`.

In [ ]:
def random_exemplars(cfg: DatasetConfig, train_df: pd.DataFrame, k: int, rng: random.Random) -> List[Tuple[str, str]]:
    idx = rng.sample(range(len(train_df)), k)
    out = []
    for i in idx:
        smi = train_df["smiles"].iloc[i]
        lab = label_for_prompt(cfg, train_df["label"].iloc[i])
        out.append((smi, lab))
    return out

def retrieval_exemplars(cfg: DatasetConfig, index: TanimotoIndex, test_smiles: str, k: int) -> List[Tuple[str, str]]:
    hits = index.topk(test_smiles, k=k)
    return [(smi, label_for_prompt(cfg, lab)) for smi, lab, _ in hits]

# Demo
rng = random.Random(SEED)
print("Random 3-shot:")
for smi, lab in random_exemplars(DATASETS["BBBP"], bbbp_train, k=3, rng=rng):
    print(f"  {lab:4s}  {smi[:80]}")
print("\nRetrieval 3-shot for test[0]:")
for smi, lab in retrieval_exemplars(DATASETS["BBBP"], bbbp_index, bbbp_test["smiles"].iloc[0], k=3):
    print(f"  {lab:4s}  {smi[:80]}")

## 8. Smoke test: GPT-2 + 10 BBBP test rows

Loads GPT-2 (124M, fast) and runs zero / few / retrieval on a tiny BBBP subset. If this works, 04b and 04c just scale it up to full test sets and bigger models.

**What to look for:** all three regimes complete without error and produce a probability in [0, 1] for each row. The AUC values will be near-random (GPT-2 knows almost no chemistry) — that's expected and the *point* of having a floor baseline.

In [ ]:
MODEL_ID = "gpt2"   # 124M, ships fast, no auth needed

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(DEVICE).eval()
print("Loaded:", MODEL_ID, "| params:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

In [ ]:
N_SMOKE = 10
K       = 5
cfg     = DATASETS["BBBP"]

test_subset = bbbp_test.head(N_SMOKE).reset_index(drop=True)
rng = random.Random(SEED)

def evaluate_regime(regime: str):
    pos_idx = cfg.label_words.index(cfg.positive_word)
    probs_pos, y_true = [], []
    for _, row in test_subset.iterrows():
        smi, y = row["smiles"], int(row["label"])
        if regime == "zero":
            exemplars = None
        elif regime == "few":
            exemplars = random_exemplars(cfg, bbbp_train, k=K, rng=rng)
        elif regime == "retrieval":
            exemplars = retrieval_exemplars(cfg, bbbp_index, smi, k=K)
        prompt = build_prompt(cfg, smi, exemplars)
        probs = score_labels(model, tokenizer, prompt, cfg.label_words)
        probs_pos.append(probs[pos_idx])
        y_true.append(y)
    probs_pos = np.array(probs_pos)
    y_true = np.array(y_true)
    # Can only compute AUC if both classes present in the tiny smoke sample
    auc = roc_auc_score(y_true, probs_pos) if len(set(y_true)) == 2 else float("nan")
    print(f"  {regime:9s}  AUC = {auc:.4f}   P(yes) range = [{probs_pos.min():.3f}, {probs_pos.max():.3f}]")

print(f"Smoke test: GPT-2 on first {N_SMOKE} BBBP test rows")
evaluate_regime("zero")
evaluate_regime("few")
evaluate_regime("retrieval")

## 9. Done

**What we validated:**
- ✅ Prompt template renders correctly
- ✅ Logit scoring returns sane probabilities in [0, 1]
- ✅ Tanimoto retrieval finds plausibly similar molecules
- ✅ All three prompting regimes execute end-to-end

**What we did NOT do:**
- ❌ Evaluate on full test sets (only 10 BBBP rows)
- ❌ Use the bigger models (only GPT-2)
- ❌ Write to `baselines.csv` (04b/04c will append properly)

**Next (04b):** scale this up — GPT-2 + SmolLM3-3B on all three datasets × all three regimes (18 evaluations). Then 04c does Phi-4-mini alone (9 evaluations). Final scoreboard has 6 + 27 = 33 rows.